In [ ]:
!pip install -q kagglehub

import kagglehub
kagglehub.login()

In [ ]:
path = kagglehub.dataset_download("nih-chest-xrays/sample")
print("Dataset downloaded to:", path)

In [ ]:
import os

csv_candidates = [os.path.join(root, f) for root, _, files in os.walk(path) for f in files if f.endswith(".csv")]
img_dir_candidates = [root for root, dirs, files in os.walk(path) if any(f.lower().endswith(".png") for f in files)]

print("CSV found:", csv_candidates)
print("Image dirs found:", img_dir_candidates)

In [ ]:
import pandas as pd

labels_csv_path = csv_candidates[0]
IMAGES_DIR = img_dir_candidates[0]

labels_df = pd.read_csv(labels_csv_path)
print(labels_df.shape)

all_labels = labels_df["Finding Labels"].str.split("|").explode()
label_counts = all_labels.value_counts()
print(label_counts)
print()
print(f"No Finding: {(labels_df['Finding Labels'] == 'No Finding').mean():.1%} of images")

In [ ]:
import numpy as np

CONDITIONS = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass",
    "Nodule", "Pneumonia", "Pneumothorax", "Consolidation", "Edema",
    "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia"
]

# Catches a casing/naming mismatch BEFORE training on silently-wrong labels.
missing = [c for c in CONDITIONS if c not in label_counts.index]
if missing:
    print("WARNING: these condition names don't match the dataset's labels:", missing)
    print("Available labels:", sorted(label_counts.index.tolist()))
else:
    print("All 14 condition names match the dataset's label vocabulary.")

def labels_to_vector(finding_labels_str):
    labels = finding_labels_str.split("|")
    return np.array([1.0 if c in labels else 0.0 for c in CONDITIONS], dtype=np.float32)

label_matrix = np.stack(labels_df["Finding Labels"].apply(labels_to_vector).to_numpy())
print(label_matrix.shape)

for i, c in enumerate(CONDITIONS):
    print(f"{c}: {int(label_matrix[:, i].sum())} positive examples")

In [ ]:
from sklearn.model_selection import train_test_split

image_indices = labels_df["Image Index"].to_numpy()
train_idx, val_idx = train_test_split(np.arange(len(image_indices)), test_size=0.2, random_state=42)

train_files = image_indices[train_idx]
val_files = image_indices[val_idx]
train_labels = label_matrix[train_idx]
val_labels = label_matrix[val_idx]

print(f"Train: {len(train_files)}, Val: {len(val_files)}")

In [ ]:
import tensorflow as tf

IMG_SIZE = 128

def load_image(filename, label):
    path = tf.strings.join([IMAGES_DIR, filename], separator="/")
    image = tf.io.read_file(path)
    image = tf.io.decode_png(image, channels=3)  # handles grayscale X-rays -> RGB automatically
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = image / 255.0
    return image, label

def make_dataset(files, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((files, labels))
    if shuffle:
        ds = ds.shuffle(len(files), seed=42)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(32).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_files, train_labels, shuffle=True)
val_ds = make_dataset(val_files, val_labels)

# confirm a batch actually loads before committing to a full training run
for imgs, labs in train_ds.take(1):
    print("batch image shape:", imgs.shape, "batch label shape:", labs.shape)

In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Conv2D(16, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(len(CONDITIONS), activation="sigmoid"),  # NOT softmax — multi-label
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(multi_label=True, name="auc")],
)
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

callbacks = [EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    shuffle=False,  # dataset is already shuffled above; avoids a harmless-but-noisy warning
    callbacks=callbacks,
)

In [ ]:
from sklearn.metrics import roc_auc_score

y_true = val_labels
y_pred = model.predict(val_ds, verbose=0)

print(f"{'Condition':<20} {'AUC':>6} {'Positives in val':>18}")
aucs = []
for i, c in enumerate(CONDITIONS):
    n_pos = int(y_true[:, i].sum())
    if n_pos == 0 or n_pos == len(y_true):
        print(f"{c:<20} {'N/A':>6} {n_pos:>18}  (too few/no positive examples to compute AUC)")
        continue
    auc = roc_auc_score(y_true[:, i], y_pred[:, i])
    aucs.append(auc)
    print(f"{c:<20} {auc:>6.3f} {n_pos:>18}")

print(f"\nMean AUC across conditions with enough data: {np.mean(aucs):.3f}")

In [ ]:
import json

os.makedirs("model_artifacts", exist_ok=True)
model.save("model_artifacts/v1_baseline_cnn.keras")

with open("model_artifacts/condition_names.json", "w") as f:
    json.dump(CONDITIONS, f, indent=2)

print("Artifacts written:")
!ls -la model_artifacts

In [ ]:
import shutil

shutil.make_archive("v1_imaging_artifacts", "zip", "model_artifacts")

from google.colab import files
files.download("v1_imaging_artifacts.zip")